# SVD-based Movie Recommender

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path

In [ ]:
# Diem so 5 phim dung de 'bat mach' so thich (MovieID, Rating)
ANCHOR_MOVIES = [
    (1, 5.0),    # Toy Story
    (2571, 4.0), # The Matrix
    (318, 5.0),  # Shawshank Redemption
    (480, 5.0),  # Jurassic Park
    (593, 5.0),  # Silence of the Lambs
]

# 5 phim dung de kiem chung du doan (MovieID)
TEST_MOVIES = [260, 589, 356, 2858, 1196]

# Co the doi k de thu nghiem
k = 50

## Data path

Notebook tu tim du lieu theo thu tu: `./data`, `/content/SVD/data`, `/content/data`.
Neu khong tim thay, hay upload thu muc `data` len Colab truoc khi chay tiep.

In [ ]:
candidate_dirs = [
    Path('data'),
    Path('/content/SVD/data'),
    Path('/content/data'),
]

data_dir = None
for d in candidate_dirs:
    if (d / 'ratings.csv').exists() and (d / 'movies.csv').exists():
        data_dir = d
        break

if data_dir is None:
    raise FileNotFoundError(
        'Khong tim thay data/ratings.csv va data/movies.csv. '
        'Hay upload thu muc data len Colab.'
    )

print(f'Dang tai du lieu tu: {data_dir.resolve()}')
data_ratings = pd.read_csv(data_dir / 'ratings.csv')
data_movies = pd.read_csv(data_dir / 'movies.csv')

movies = data_ratings['movieId'].unique()
users = data_ratings['userId'].unique()

movie_to_idx = {res: idx for idx, res in enumerate(movies)}
idx_to_movie = {idx: res for idx, res in enumerate(movies)}
user_to_idx = {res: idx for idx, res in enumerate(users)}

In [ ]:
# Khoi tao ma tran rating
ratings_mat = np.zeros((len(movies), len(users)), dtype=np.float32)
for row in data_ratings.itertuples():
    ratings_mat[movie_to_idx[row.movieId], user_to_idx[row.userId]] = row.rating

# Mean centering
sums = ratings_mat.sum(axis=1)
counts = (ratings_mat != 0).sum(axis=1)
movie_means = sums / (counts + 1e-9)

normalised_mat = ratings_mat - movie_means.reshape(-1, 1)
normalised_mat[ratings_mat == 0] = 0

print('Ma tran rating:', ratings_mat.shape)

In [ ]:
print('Dang phan ra SVD...')
U, S, V = np.linalg.svd(normalised_mat, full_matrices=False)
movie_features = U[:, :k]

print('Top 5 singular values:', S[:5])

In [ ]:
print(f'[BUOC 1] Tim vector so thich tu {len(ANCHOR_MOVIES)} phim:')
A_matrix = []
b_vector = []

for m_id, rating in ANCHOR_MOVIES:
    if m_id not in movie_to_idx:
        raise ValueError(f'MovieID {m_id} khong co trong ratings.csv')

    title_rows = data_movies[data_movies.movieId == m_id]
    title = title_rows.title.values[0] if not title_rows.empty else f'Unknown ID {m_id}'
    print(f'  - {rating} sao : {title}')

    idx_m = movie_to_idx[m_id]
    A_matrix.append(movie_features[idx_m, :])
    b_vector.append(rating - movie_means[idx_m])

A_matrix = np.array(A_matrix)
b_vector = np.array(b_vector)

# Pseudoinverse giai he A*x=b
user_vector = np.linalg.pinv(A_matrix) @ b_vector
print('Da suy ra user_vector, shape =', user_vector.shape)

In [ ]:
print(f'\n[BUOC 2] Du doan diem cho {len(TEST_MOVIES)} phim kiem chung:')
for m_id in TEST_MOVIES:
    if m_id not in movie_to_idx:
        print(f'  -> Bo qua MovieID {m_id}: khong co trong ratings.csv')
        continue

    title_rows = data_movies[data_movies.movieId == m_id]
    title = title_rows.title.values[0] if not title_rows.empty else f'Unknown ID {m_id}'
    idx_m = movie_to_idx[m_id]

    pred_centered = np.dot(movie_features[idx_m, :], user_vector)
    pred_rating = np.clip(pred_centered + movie_means[idx_m], 1.0, 5.0)

    print(f'  -> {pred_rating:.1f} sao : {title}')